# Notebook 02 -- Preprocesamiento y Feature Engineering

---


## Objetivo

Transformar serie bruta en dataset enriquecido: detección de outliers (STL), imputación adaptativa, ingeniería de 38 variables (calendario, Fourier, rezagos, móviles). Garantizar causalidad (sin data leakage).


## Entrada

data/raw/demand.parquet


## Salida

data/processed/features.parquet (52,608 × 39) + data/processed/feature_names.json


---
## Instalación

In [ ]:
# Instalación de dependencias
# En Docker (make up / make pipeline) ya vienen instaladas con versiones fijas
# (requirements.txt). En Google Colab, después de clonar el repo y hacer
# %cd notebooks, se instalan las mismas versiones:
import sys
if "google.colab" in sys.modules:
    !pip install -q -r ../requirements.txt

---
## Configuración

In [ ]:
import os
# ============================================================
# CONFIGURACIÓN — Modificar según entorno
# ============================================================
DATA_DIR = "../data"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"
# Figuras en inglés (versión para publicación); la carpeta no se versiona
os.makedirs(f"{RESULTS_DIR}/english", exist_ok=True)
RANDOM_SEED  = 42
N_JOBS       = -1

---
## Imports

In [ ]:
from __future__ import annotations

import json
import os
import warnings
from pathlib import Path

import holidays
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import interpolate as sp_interp
from scipy import stats as sp_stats
from statsmodels.tsa.seasonal import STL, MSTL
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Configuración predeterminada de Matplotlib ─────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.figsize": (14, 4),
    "figure.dpi": 120,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "font.family": "serif",
    "axes.grid": True,
    "grid.alpha": 0.3,
})
sns.set_palette("colorblind")

# ── Asegurar que existan los directorios de salida ─────────────────────────────────────────
Path(DATA_DIR, "processed").mkdir(parents=True, exist_ok=True)
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# ── Cargar demanda en bruto ─────────────────────────────────────────────────────────
raw_path = Path(DATA_DIR, "raw", "demand.parquet")
df = pd.read_parquet(raw_path)
print(f"Forma original: {df.shape}")
print(f"Rango de fechas: {df.index.min()} → {df.index.max()}" if isinstance(df.index, pd.DatetimeIndex) else "")
df.head()

---
## Carga de datos

In [ ]:
# ── Asegurar índice datetime ───────────────────────────────────────────────────
# Adaptar nombres de columnas si es necesario — esperamos una columna de fecha/hora y una columna de demanda.
if not isinstance(df.index, pd.DatetimeIndex):
    # Intentar encontrar una columna datetime
    date_cols = [c for c in df.columns if "date" in c.lower() or "fecha" in c.lower() or "time" in c.lower()]
    if date_cols:
        df[date_cols[0]] = pd.to_datetime(df[date_cols[0]])
        df = df.set_index(date_cols[0]).sort_index()
    else:
        raise ValueError("No se encontró una columna datetime. Ajusta la lógica de carga.")

# Identificar la columna de demanda (búsqueda insensible a mayúsculas/minúsculas)
demand_col_candidates = [c for c in df.columns if "demand" in c.lower() or "demanda" in c.lower()]
if len(demand_col_candidates) == 0:
    # Respaldo: usar la primera columna numérica
    demand_col_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
DEMAND_COL = demand_col_candidates[0]
print(f"Usando la columna de demanda: '{DEMAND_COL}'")

# Renombrar para mantener consistencia
df = df.rename(columns={DEMAND_COL: "demand"})
DEMAND_COL = "demand"

# Conservar solo la columna de demanda para esta etapa de preprocesamiento
demand_series = df["demand"].copy()
demand_series.index.name = "datetime"
demand_series = demand_series.asfreq("h")  # asegurar frecuencia horaria

print(f"\nLongitud de la serie : {len(demand_series):,} horas")
print(f"Valores faltantes    : {demand_series.isna().sum():,} ({demand_series.isna().mean()*100:.2f} %)")
print(f"Rango de fechas      : {demand_series.index.min()} → {demand_series.index.max()}")

---
# 1. Ingeniería de Variables (Solo Variables para Producción)

Cada variable calculada aquí **debe estar disponible en tiempo de inferencia**. Esto significa:
- Las variables de calendario y Fourier son determinísticas (se conocen para cualquier instante futuro).
- Las variables autorregresivas usan solo valores **pasados** (causalidad estricta).
- Las estadísticas móviles usan `center=False` (ventana hacia atrás).

In [ ]:
# ── 1.0  Construir DataFrame de variables ────────────────────────────────────────────

feat = pd.DataFrame({"demand": demand_series}, index=demand_series.index)
feat.index.name = "datetime"

# Vamos a recopilar los nombres de las variables por categoría para la exportación JSON.
feature_names: dict[str, list[str]] = {
    "target": ["demand"],
    "calendar": [],
    "holiday": [],
    "fourier": [],
    "autoregressive_lags": [],
    "autoregressive_rolling": [],
}

### 1.1 Variables de Calendario (Codificación Cíclica)

In [ ]:
# ── 1.1  Variables de calendario — codificación cíclica seno/coseno ─────────────────────────

feat["hour_sin"]      = np.sin(2 * np.pi * feat.index.hour / 24)
feat["hour_cos"]      = np.cos(2 * np.pi * feat.index.hour / 24)
feat["dayofweek_sin"] = np.sin(2 * np.pi * feat.index.dayofweek / 7)
feat["dayofweek_cos"] = np.cos(2 * np.pi * feat.index.dayofweek / 7)
feat["month_sin"]     = np.sin(2 * np.pi * feat.index.month / 12)
feat["month_cos"]     = np.cos(2 * np.pi * feat.index.month / 12)

calendar_cols = ["hour_sin", "hour_cos", "dayofweek_sin", "dayofweek_cos", "month_sin", "month_cos"]
feature_names["calendar"].extend(calendar_cols)

print(f"Variables de calendario agregadas: {calendar_cols}")

### 1.2 Festivos en Colombia (Ley 51 de 1983)

La *Ley 51 de 1983* en Colombia traslada ciertos festivos al lunes siguiente
("puentes festivos"). La librería `holidays.Colombia()` gestiona esto.
Adicionalmente, detectamos **días puente** — viernes o sábados adyacentes a un
festivo en lunes, que típicamente presentan una demanda reducida.

In [ ]:
# ── 1.2  Variables de festivos ────────────────────────────────────────────────────

# Construir conjunto de festivos que cubra todo el rango de fechas + margen de 1 año
year_min = feat.index.year.min() - 1
year_max = feat.index.year.max() + 1
co_holidays = holidays.Colombia(years=range(year_min, year_max + 1))
holiday_dates = set(co_holidays.keys())

# is_holiday
# feat.index.date es un ndarray: se envuelve en una Serie para usar .isin()
feat["is_holiday"] = pd.Series(feat.index.date, index=feat.index).isin(
    holiday_dates
).astype(int)

# is_bridge: viernes antes de un festivo en lunes, o sábado entre festivo y domingo
dates_series = pd.Series(feat.index.date).drop_duplicates().reset_index(drop=True)
bridge_dates = set()
for hd in holiday_dates:
    hd_pd = pd.Timestamp(hd)
    # Si el festivo cae en lunes, el viernes y sábado anteriores son días puente
    if hd_pd.weekday() == 0:  # Monday
        bridge_dates.add((hd_pd - pd.Timedelta(days=3)).date())  # Friday
        bridge_dates.add((hd_pd - pd.Timedelta(days=2)).date())  # Saturday
    # Si el festivo cae en viernes, el sábado es puente
    elif hd_pd.weekday() == 4:  # Friday
        bridge_dates.add((hd_pd + pd.Timedelta(days=1)).date())  # Saturday

feat["is_bridge"] = pd.Series(
    feat.index.date, index=feat.index
).isin(bridge_dates).astype(int)

# days_to_next_holiday / days_from_prev_holiday
sorted_holidays = sorted(holiday_dates)
holiday_ts = pd.to_datetime(sorted_holidays)


def days_to_nearest(dt, holiday_list, direction="next") -> int:
    """Calcula los días hasta el siguiente festivo o desde el festivo anterior."""
    d = dt.date() if hasattr(dt, "date") else dt
    d = pd.Timestamp(d)
    if direction == "next":
        future = holiday_list[holiday_list >= d]
        return (future[0] - d).days if len(future) > 0 else 365
    else:
        past = holiday_list[holiday_list <= d]
        return (d - past[-1]).days if len(past) > 0 else 365


# Calcular por fecha única y luego mapear de regreso (mucho más rápido que por fila)
unique_dates = pd.Series(feat.index.date).unique()
days_next = {d: days_to_nearest(d, holiday_ts, "next") for d in unique_dates}
days_prev = {d: days_to_nearest(d, holiday_ts, "prev") for d in unique_dates}

feat["days_to_next_holiday"]   = pd.Series(feat.index.date, index=feat.index).map(days_next).astype(int)
feat["days_from_prev_holiday"] = pd.Series(feat.index.date, index=feat.index).map(days_prev).astype(int)

# Limitar a un máximo razonable para evitar valores muy grandes
feat["days_to_next_holiday"]   = feat["days_to_next_holiday"].clip(upper=30)
feat["days_from_prev_holiday"] = feat["days_from_prev_holiday"].clip(upper=30)

holiday_cols = ["is_holiday", "is_bridge", "days_to_next_holiday", "days_from_prev_holiday"]
feature_names["holiday"].extend(holiday_cols)

print(f"Variables de festivos agregadas: {holiday_cols}")
print(f"Total de festivos en el rango: {feat['is_holiday'].sum() // 24} días")
print(f"Total de días puente: {feat['is_bridge'].sum() // 24} días")

### 1.4 Variables Autorregresivas — Rezagos

El conjunto de rezagos seleccionado cubre:
- **Horas recientes**: 1, 2, 3 (momento de corto plazo)
- **Misma hora de ayer ± 1**: 23, 24, 25 (ciclo diario)
- **Misma hora de hace dos días ± 1**: 47, 48, 49
- **Misma hora de la semana pasada ± 1**: 167, 168, 169 (ciclo semanal)

In [ ]:
# ── 1.3b  Términos de Fourier ───────────────────────────────────────────────────
# Capturan la estacionalidad diaria (24 h, 3 armónicos) y semanal (168 h, 2 armónicos).
# t = horas transcurridas desde el inicio de la serie.

FOURIER_TERMS = {24: 3, 168: 2}  # período (h) → número de armónicos

t = np.arange(len(feat))
for period, n_harmonics in FOURIER_TERMS.items():
    for k in range(1, n_harmonics + 1):
        for fname, func in (("sin", np.sin), ("cos", np.cos)):
            col_name = f"fourier_{period}_{fname}_{k}"
            feat[col_name] = func(2 * np.pi * k * t / period)
            feature_names["fourier"].append(col_name)

print(f"Términos de Fourier agregados ({len(feature_names['fourier'])}): {feature_names['fourier']}")

In [ ]:
# ── 1.4  Variables de rezagos ───────────────────────────────────────────────────────

LAG_SET = [1, 2, 3, 23, 24, 25, 47, 48, 49, 167, 168, 169]

for lag in LAG_SET:
    col_name = f"demand_lag_{lag}"
    feat[col_name] = feat["demand"].shift(lag)
    feature_names["autoregressive_lags"].append(col_name)

print(f"Variables de rezagos agregadas ({len(LAG_SET)}): rezagos = {LAG_SET}")

### 1.5 Estadísticas Móviles

**Crítico:** `center=False` (hacia atrás) para evitar data leakage.

In [ ]:
# ── 1.5  Media / desviación estándar móvil — SIEMPRE center=False ────────────────────────────

ROLLING_WINDOWS = [24, 168, 336]

for w in ROLLING_WINDOWS:
    mean_col = f"demand_rmean_{w}"
    std_col  = f"demand_rstd_{w}"
    # shift(1): la ventana termina en t-1. Sin el shift incluiría la demanda de la
    # hora que se quiere predecir (fuga del target hacia las variables).
    past_demand = feat["demand"].shift(1)
    feat[mean_col] = past_demand.rolling(window=w, min_periods=1, center=False).mean()
    feat[std_col]  = past_demand.rolling(window=w, min_periods=1, center=False).std()
    feature_names["autoregressive_rolling"].extend([mean_col, std_col])

print(f"Variables móviles agregadas: ventanas = {ROLLING_WINDOWS} (media + desviación estándar en cada una)")

In [ ]:
# ── 1.6  Resumen de variables ────────────────────────────────────────────────────

total_features = sum(len(v) for v in feature_names.values()) - 1  # excluir target
print(f"\nTotal de variables: {total_features}")
print("\nVariables por categoría:")
for cat, names in feature_names.items():
    print(f"  {cat:30s} → {len(names):3d} variables")

print(f"\nForma del DataFrame: {feat.shape}")
print(f"NaNs por columna (top 10):")
na_counts = feat.isna().sum().sort_values(ascending=False)
print(na_counts[na_counts > 0].head(10).to_string())

---
# 2. Análisis de Estacionariedad

Evaluamos la estacionariedad de la serie de demanda limpia usando dos pruebas complementarias:

| Prueba | H₀ | Interpretación |
|--------|----|----------------|
| **ADF** (Augmented Dickey-Fuller) | La serie tiene raíz unitaria (no estacionaria) | Rechazar H₀ → estacionaria |
| **KPSS** | La serie es estacionaria | Rechazar H₀ → no estacionaria |

Usar ambas pruebas en conjunto evita la ambigüedad de una sola prueba.
Luego aplicamos **descomposición MSTL** (Multi-Seasonal-Trend usando LOESS)
con períodos 24 y 168 para inspeccionar visualmente si el componente residual
es aproximadamente estacionario.

In [ ]:
# ── 2.1  Pruebas ADF y KPSS ─────────────────────────────────────────────────

def stationarity_tests(series: pd.Series, name: str = "Series") -> pd.DataFrame:
    """Ejecuta las pruebas ADF y KPSS, y retorna un DataFrame resumen."""
    results = []

    # Prueba ADF
    adf_stat, adf_pval, adf_lags, adf_nobs, adf_crit, _ = adfuller(series.dropna(), autolag="AIC")
    results.append({
        "Test": "ADF",
        "Statistic": f"{adf_stat:.4f}",
        "p-value": f"{adf_pval:.6f}",
        "Lags used": adf_lags,
        "H₀": "Raíz unitaria (no estacionaria)",
        "Conclusion": "Stationary" if adf_pval < 0.05 else "Non-stationary",
    })

    # Prueba KPSS
    kpss_stat, kpss_pval, kpss_lags, kpss_crit = kpss(series.dropna(), regression="c", nlags="auto")
    results.append({
        "Test": "KPSS",
        "Statistic": f"{kpss_stat:.4f}",
        "p-value": f"{kpss_pval:.6f}",
        "Lags used": kpss_lags,
        "H₀": "Estacionaria",
        "Conclusion": "Stationary" if kpss_pval > 0.05 else "Non-stationary",
    })

    df_results = pd.DataFrame(results)
    print(f"\n{'=' * 70}")
    print(f"  Pruebas de Estacionariedad — {name}")
    print(f"{'=' * 70}")
    print(df_results.to_string(index=False))

    # Interpretación conjunta
    adf_stationary = adf_pval < 0.05
    kpss_stationary = kpss_pval > 0.05
    if adf_stationary and kpss_stationary:
        print("\n  → Ambas pruebas coinciden: la serie es ESTACIONARIA.")
    elif not adf_stationary and not kpss_stationary:
        print("\n  → Ambas pruebas coinciden: la serie es NO ESTACIONARIA.")
    elif adf_stationary and not kpss_stationary:
        print("\n  → Conflicto: ADF dice estacionaria, KPSS dice no estacionaria.")
        print("    Probablemente es estacionaria en tendencia (estacionaria alrededor de una tendencia determinística).")
    else:
        print("\n  → Conflicto: ADF dice no estacionaria, KPSS dice estacionaria.")
        print("    Probablemente es estacionaria en diferencias.")
    print(f"{'=' * 70}")

    return df_results


# Prueba sobre la demanda imputada en bruto
_ = stationarity_tests(demand_series, name="Demand")

In [ ]:
# ── 2.2  Descomposición MSTL (periods = [24, 168]) ──────────────────────────

mstl = MSTL(demand_series.dropna(), periods=[24, 168], stl_kwargs={"robust": True})
mstl_result = mstl.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

components = [
    ("Observada", mstl_result.observed, "steelblue"),
    ("Tendencia", mstl_result.trend, "darkorange"),
    ("Estacional (combinada)", mstl_result.seasonal.sum(axis=1) if mstl_result.seasonal.ndim > 1 else mstl_result.seasonal, "seagreen"),
    ("Residual", mstl_result.resid, "indianred"),
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.4, color=color)
    ax.set_ylabel("kWh")
    ax.set_title(f"MSTL — {title}", fontsize=11)

axes[-1].set_xlabel("Fecha")
fig.suptitle("Descomposición Multiestacional-Tendencia (P=24, P=168)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(Path(RESULTS_DIR, "02_mstl_decomposition.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
demand_series

In [ ]:
# ── 2.2  Descomposición MSTL (periods = [24, 168]) ──────────────────────────
df_2022_jan = demand_series[(demand_series.index.year == 2022) & (demand_series.index.month == 1)].copy()

mstl = MSTL(df_2022_jan.dropna(), periods=[24, 168], stl_kwargs={"robust": True})
mstl_result = mstl.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

components = [
    ("Observada", mstl_result.observed, "steelblue"),
    ("Tendencia", mstl_result.trend, "darkorange"),
    ("Estacional (combinada)", mstl_result.seasonal.sum(axis=1) if mstl_result.seasonal.ndim > 1 else mstl_result.seasonal, "seagreen"),
    ("Residual", mstl_result.resid, "indianred"),
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.4, color=color)
    ax.set_ylabel("kWh")
    ax.set_title(f"MSTL — {title}", fontsize=11)

axes[-1].set_xlabel("Fecha")
fig.suptitle("Descomposición Multiestacional-Tendencia (P=24, P=168)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(Path(RESULTS_DIR, "02_mstl_decomposition_filtrado.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
mstl = MSTL(demand_series.dropna(), periods=[24, 168], stl_kwargs={"robust": True})
mstl_result = mstl.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

components = [
    ("Observed", mstl_result.observed, "steelblue"),
    ("Trend", mstl_result.trend, "darkorange"),
    ("Seasonal (combined)", mstl_result.seasonal.sum(axis=1) if mstl_result.seasonal.ndim > 1 else mstl_result.seasonal, "seagreen"),
    ("Residual", mstl_result.resid, "indianred"),
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.4, color=color)
    ax.set_ylabel("kWh")
    ax.set_title(f"MSTL — {title}", fontsize=11)

axes[-1].set_xlabel("Date")
fig.suptitle("Multi-Seasonal-Trend Decomposition (P=24, P=168)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(Path(RESULTS_DIR, "english", "02_mstl_decomposition.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 2.3  Prueba de estacionariedad sobre el residual MSTL ────────────────────────────────

mstl_resid = mstl_result.resid.dropna()
_ = stationarity_tests(mstl_resid, name="Residual MSTL")

# Verificación visual: distribución del residual
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.hist(mstl_resid.values, bins=100, density=True, alpha=0.7, color="indianred", edgecolor="white")
xmin, xmax = ax.get_xlim()
x = np.linspace(xmin, xmax, 200)
ax.plot(x, sp_stats.norm.pdf(x, mstl_resid.mean(), mstl_resid.std()),
        "k--", linewidth=1.5, label="Ajuste normal")
ax.set_title("Distribución del Residual MSTL")
ax.set_xlabel("Residual (kWh)")
ax.legend()

ax = axes[1]
sp_stats.probplot(mstl_resid.values, dist="norm", plot=ax)
ax.set_title("Gráfico Q-Q — Residual MSTL vs Normal")

plt.tight_layout()
plt.savefig(Path(RESULTS_DIR, "02_mstl_residual_diagnostics.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
mstl_resid = mstl_result.resid.dropna()
_ = stationarity_tests(mstl_resid, name="MSTL Residual")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.hist(mstl_resid.values, bins=100, density=True, alpha=0.7, color="indianred", edgecolor="white")
xmin, xmax = ax.get_xlim()
x = np.linspace(xmin, xmax, 200)
ax.plot(x, sp_stats.norm.pdf(x, mstl_resid.mean(), mstl_resid.std()),
        "k--", linewidth=1.5, label="Normal fit")
ax.set_title("MSTL Residual Distribution")
ax.set_xlabel("Residual (kWh)")
ax.legend()

ax = axes[1]
sp_stats.probplot(mstl_resid.values, dist="norm", plot=ax)
ax.set_title("Q-Q Plot — MSTL Residual vs Normal")

plt.tight_layout()
plt.savefig(Path(RESULTS_DIR, "english", "02_mstl_residual_diagnostics.png"), dpi=150, bbox_inches="tight")
plt.show()

---
# 3. Guardar archivos

Guardamos dos artefactos:

1. **`data/processed/features.parquet`** — el DataFrame completo de variables
   (incluyendo `demand` como la columna objetivo). Las filas con NaN debidas al
   calentamiento inicial de rezagos/ventanas móviles *se conservan* (se eliminarán en el entrenamiento).

2. **`data/processed/feature_names.json`** — un diccionario que asigna cada
   categoría de variables a sus nombres de columna, para la configuración posterior del modelo.

In [ ]:
# ── 3.1  Guardar features.parquet ──────────────────────────────────────────────

out_path = Path(DATA_DIR, "processed", "features.parquet")
feat.to_parquet(out_path, engine="pyarrow")
print(f"Guardado: {out_path}")
print(f"  Forma     : {feat.shape}")
print(f"  Columnas  : {feat.shape[1]}")
print(f"  Tamaño    : {out_path.stat().st_size / 1024**2:.1f} MB")

In [ ]:
# ── 3.2  Guardar feature_names.json ────────────────────────────────────────────

json_path = Path(DATA_DIR, "processed", "feature_names.json")
with open(json_path, "w") as f:
    json.dump(feature_names, f, indent=2)

print(f"Guardado: {json_path}")
print(json.dumps(feature_names, indent=2))

In [ ]:
# ── 3.3  Resumen final ──────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("  PREPROCESAMIENTO COMPLETO")
print("=" * 70)
print(f"  Datos originales        : {len(demand_series):,}")
print(f"  Variables generadas     : {total_features}")
print(f"  Filas de salida         : {len(feat):,}")
print(f"  Columnas de salida      : {feat.shape[1]}")
print(f"\n  Archivos:")
print(f"    → {out_path}")
print(f"    → {json_path}")
print("=" * 70)